# 03 - TF-IDF Keyword Extraction

This notebook uses TF-IDF (Term Frequency-Inverse Document Frequency) to:
- Extract the most important keywords from CVE descriptions
- Identify common vulnerability terms and attack patterns
- Visualize the most significant terms
- Build a TF-IDF feature matrix for downstream models

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import joblib
import sys
sys.path.append('..')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## 1. Load Preprocessed Data

In [2]:
df = pd.read_csv('../data/cve_preprocessed.csv')
print(f"Loaded {len(df)} CVE records")

# Use cleaned descriptions for TF-IDF
texts = df['Cleaned_Description'].fillna('').tolist()
print(f"Non-empty descriptions: {sum(1 for t in texts if t.strip())}")

Loaded 1314 CVE records
Non-empty descriptions: 1314


## 2. Build TF-IDF Matrix

In [3]:
# Configure TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,      # Limit features for performance
    min_df=2,               # Must appear in at least 2 documents
    max_df=0.95,            # Ignore terms in >95% of documents
    ngram_range=(1, 2),     # Unigrams and bigrams
    sublinear_tf=True       # Apply sublinear TF scaling
)

tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"  Documents: {tfidf_matrix.shape[0]}")
print(f"  Features (terms): {tfidf_matrix.shape[1]}")
print(f"  Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")

TF-IDF Matrix shape: (1314, 5000)
  Documents: 1314
  Features (terms): 5000
  Sparsity: 0.9916


## 3. Top Global TF-IDF Terms

In [4]:
# Calculate mean TF-IDF score across all documents
mean_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()
top_indices = mean_tfidf.argsort()[::-1][:30]

top_terms = [(feature_names[i], mean_tfidf[i]) for i in top_indices]

print("Top 30 Terms by Mean TF-IDF Score:")
print("=" * 50)
for term, score in top_terms:
    print(f"  {term:30s} {score:.4f}")

# Visualization
fig = px.bar(x=[t[1] for t in top_terms], y=[t[0] for t in top_terms],
             orientation='h',
             labels={'x': 'Mean TF-IDF Score', 'y': 'Term'},
             title='Top 30 Terms by Mean TF-IDF Score Across All CVEs',
             color=[t[1] for t in top_terms],
             color_continuous_scale='Reds')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700, showlegend=False)
fig.show()

Top 30 Terms by Mean TF-IDF Score:
  vulnerability                  0.0413
  version                        0.0372
  file                           0.0251
  issue                          0.0235
  attacker                       0.0228
  user                           0.0212
  may                            0.0210
  via                            0.0200
  lead                           0.0198
  code                           0.0192
  allows                         0.0175
  arbitrary                      0.0167
  remote                         0.0150
  used                           0.0149
  affect                         0.0147
  access                         0.0144
  exploit                        0.0144
  service                        0.0143
  attack                         0.0143
  site                           0.0142
  found                          0.0142
  function                       0.0141
  injection                      0.0140
  version version                0.0139
  exe

## 4. Extract Top Keywords per CVE

In [5]:
def get_top_keywords(tfidf_row, feature_names, n=5):
    """Extract top-n keywords from a single document's TF-IDF vector."""
    row_array = tfidf_row.toarray().flatten()
    top_indices = row_array.argsort()[::-1][:n]
    return [(feature_names[i], row_array[i]) for i in top_indices if row_array[i] > 0]

# Extract keywords for first 10 CVEs
print("Top Keywords per CVE (sample):")
print("=" * 80)
for i in range(min(10, len(df))):
    keywords = get_top_keywords(tfidf_matrix[i], feature_names, n=5)
    kw_str = ', '.join([f"{kw[0]} ({kw[1]:.3f})" for kw in keywords])
    print(f"\n{df['CVE ID'].iloc[i]}:")
    print(f"  Keywords: {kw_str}")

Top Keywords per CVE (sample):

CVE-2024-21732:
  Keywords: flycms (0.461), allows xss (0.428), xss via (0.420), feature (0.357), permission (0.294)

CVE-2023-5877:
  Keywords: affiliate (0.348), ssrf issue (0.222), rfc (0.222), visitor (0.213), forgery ssrf (0.213)

CVE-2023-6000:
  Keywords: popup builder (0.264), lead stored (0.264), injecting (0.264), visitor (0.254), raw (0.245)

CVE-2023-6037:
  Keywords: plugin sanitise (0.190), slider (0.182), disallowed example (0.176), capability disallowed (0.176), html capability (0.176)

CVE-2023-6064:
  Keywords: automatically (0.266), containing sensitive (0.266), file containing (0.255), publicly accessible (0.255), payment gateway (0.255)

CVE-2023-6113:
  Keywords: backup (0.463), plugin version (0.273), leaking (0.251), ongoing (0.251), visitor (0.241)

CVE-2023-6271:
  Keywords: backup (0.365), store progress (0.225), progress backup (0.225), monitoring (0.225), find (0.225)

CVE-2023-6421:
  Keywords: download (0.367), leaking (0.2

In [6]:
# Add top keywords column to dataframe
all_keywords = []
for i in range(len(df)):
    keywords = get_top_keywords(tfidf_matrix[i], feature_names, n=5)
    kw_list = [kw[0] for kw in keywords]
    all_keywords.append(', '.join(kw_list))

df['Top_Keywords'] = all_keywords
df[['CVE ID', 'Top_Keywords']].head(10)

,CVE ID,Top_Keywords
0,CVE-2024-21732,"flycms, allows xss, xss via, feature, permission"
1,CVE-2023-5877,"affiliate, ssrf issue, rfc, visitor, forgery ssrf"
2,CVE-2023-6000,"popup builder, lead stored, injecting, visitor..."
3,CVE-2023-6037,"plugin sanitise, slider, disallowed example, c..."
4,CVE-2023-6064,"automatically, containing sensitive, file cont..."
5,CVE-2023-6113,"backup, plugin version, leaking, ongoing, visitor"
6,CVE-2023-6271,"backup, store progress, progress backup, monit..."
7,CVE-2023-6421,"download, leaking, manager wordpress, upon, do..."
8,CVE-2023-6485,"player, user low, admins, around, perform stored"
9,CVE-2024-0181,"admin, admin admin, admin panel, panel manipul..."


## 5. Keywords by Vulnerability Type

In [7]:
# Analyze top terms per vulnerability type
vuln_types = df['Vulnerability_Type'].unique()
vuln_types = [v for v in vuln_types if v != 'Other']

print("Top Keywords by Vulnerability Type:")
print("=" * 60)

for vtype in sorted(vuln_types):
    mask = df['Vulnerability_Type'] == vtype
    if mask.sum() < 2:
        continue
    type_tfidf = tfidf_matrix[mask]
    type_mean = np.array(type_tfidf.mean(axis=0)).flatten()
    top_idx = type_mean.argsort()[::-1][:8]
    top_terms_type = [feature_names[i] for i in top_idx if type_mean[i] > 0]
    print(f"\n{vtype}:")
    print(f"  {', '.join(top_terms_type)}")

Top Keywords by Vulnerability Type:

Authentication Bypass:
  version version, bypass, authentication, authentication bypass, version, bypass authentication, path, shiro

Buffer Overflow:
  overflow, stack, tenda, tenda version, overflow via, buffer overflow, stack overflow, buffer

CSRF:
  site request, csrf, request forgery, forgery, forgery csrf, request, site, csrf vulnerability

Command Injection:
  command, command injection, injection, injection vulnerability, arbitrary command, totolink, vulnerability, contain command

Cross-Site Scripting (XSS):
  xss, scripting, site scripting, cross, cross site, site, page, stored

Denial of Service:
  denial, denial service, service, cause, issue, crash, version, cause denial

Information Disclosure:
  information, information disclosure, disclosure, sensitive information, sensitive, disclosure vulnerability, exposure, exposure sensitive

Path Traversal:
  traversal, path traversal, path, file, directory, traversal vulnerability, lead path,

## 6. Keywords by Severity Level

In [8]:
# Top keywords by severity
for severity in ['Critical', 'High', 'Medium', 'Low']:
    mask = df['Severity'] == severity
    if mask.sum() < 2:
        continue
    sev_tfidf = tfidf_matrix[mask]
    sev_mean = np.array(sev_tfidf.mean(axis=0)).flatten()
    top_idx = sev_mean.argsort()[::-1][:10]
    top_terms_sev = [(feature_names[i], sev_mean[i]) for i in top_idx]
    print(f"\n{severity} Severity - Top Terms:")
    for term, score in top_terms_sev:
        print(f"  {term:25s} {score:.4f}")


Critical Severity - Top Terms:
  parameter                 0.0457
  via                       0.0418
  version                   0.0416
  injection vulnerability   0.0348
  command                   0.0345
  function                  0.0327
  injection                 0.0315
  vulnerability             0.0314
  overflow via              0.0293
  stack                     0.0292

High Severity - Top Terms:
  vulnerability             0.0441
  file                      0.0370
  version                   0.0365
  code                      0.0289
  execution                 0.0252
  code execution            0.0241
  arbitrary                 0.0239
  crafted                   0.0238
  attacker                  0.0225
  lead                      0.0224

Medium Severity - Top Terms:
  vulnerability             0.0402
  version                   0.0371
  issue                     0.0274
  user                      0.0258
  may                       0.0249
  attacker                  0.0242


## 7. Security-Specific Pattern Detection

In [9]:
# Search for specific attack pattern terms in the TF-IDF vocabulary
attack_patterns = [
    'remote execution', 'buffer overflow', 'sql injection',
    'denial service', 'cross site', 'privilege escalation',
    'code execution', 'command injection', 'path traversal',
    'information disclosure', 'authentication bypass'
]

print("Attack Pattern TF-IDF Presence:")
print("=" * 60)
feature_set = set(feature_names)
for pattern in attack_patterns:
    if pattern in feature_set:
        idx = list(feature_names).index(pattern)
        doc_freq = (tfidf_matrix[:, idx].toarray() > 0).sum()
        avg_score = tfidf_matrix[:, idx].toarray().mean()
        print(f"  '{pattern}': in {doc_freq} documents, avg TF-IDF: {avg_score:.4f}")
    else:
        print(f"  '{pattern}': not in vocabulary (may appear differently)")

Attack Pattern TF-IDF Presence:
  'remote execution': not in vocabulary (may appear differently)
  'buffer overflow': in 62 documents, avg TF-IDF: 0.0059
  'sql injection': in 126 documents, avg TF-IDF: 0.0105
  'denial service': in 135 documents, avg TF-IDF: 0.0117
  'cross site': in 174 documents, avg TF-IDF: 0.0125
  'privilege escalation': in 10 documents, avg TF-IDF: 0.0014
  'code execution': in 136 documents, avg TF-IDF: 0.0110
  'command injection': in 37 documents, avg TF-IDF: 0.0047
  'path traversal': in 16 documents, avg TF-IDF: 0.0020
  'information disclosure': in 45 documents, avg TF-IDF: 0.0063
  'authentication bypass': in 5 documents, avg TF-IDF: 0.0009


## 8. Save TF-IDF Artifacts

In [10]:
import scipy.sparse

# Save TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, '../models/tfidf_vectorizer.joblib')

# Save TF-IDF matrix
scipy.sparse.save_npz('../models/tfidf_matrix.npz', tfidf_matrix)

# Save updated dataframe
df.to_csv('../data/cve_with_keywords.csv', index=False)

print("Saved:")
print("  - TF-IDF vectorizer: models/tfidf_vectorizer.joblib")
print("  - TF-IDF matrix: models/tfidf_matrix.npz")
print("  - Updated dataset: data/cve_with_keywords.csv")
print("\n✅ TF-IDF keyword extraction complete!")

Saved:
  - TF-IDF vectorizer: models/tfidf_vectorizer.joblib
  - TF-IDF matrix: models/tfidf_matrix.npz
  - Updated dataset: data/cve_with_keywords.csv

✅ TF-IDF keyword extraction complete!
